In [1]:
from google.colab import files
uploaded=files.upload()

Saving ecommerce_sales_analytics_5000.csv to ecommerce_sales_analytics_5000 (1).csv


In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv("/content/ecommerce_sales_analytics_5000.csv")
df

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,1/1/2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,1/2/2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,1/3/2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,1/5/2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,14996,9/5/2035,1289,Clothing,West,2,371.74,0.27,COD,3,2.3,542.74
4996,14997,9/6/2035,1294,Beauty,North,3,553.67,0.14,COD,11,2.2,1428.47
4997,14998,9/7/2035,1450,Beauty,West,3,77.22,0.24,Wallet,6,3.4,176.06
4998,14999,9/8/2035,1903,Electronics,South,4,138.41,0.21,COD,4,2.8,437.38


In [3]:


# Create a copy of original dataset
df_copy = df.copy()

# 1. Display first 5 rows
print(df_copy.head())

# 2. Display last 5 rows
print(df_copy.tail())

# 3. Display random 10 rows
print(df_copy.sample(10))

# 4. Find rows and columns
print(df_copy.shape)

# 5. Display column names
print(df_copy.columns)

# 6. Display data types
print(df_copy.dtypes)

# 7. Find missing values
print(df_copy.isnull().sum())

# 8. Total missing values
print(df_copy.isnull().sum().sum())

# 9. Find duplicate rows
print(df_copy.duplicated().sum())

# 10. Remove duplicate rows
df_copy = df_copy.drop_duplicates()

# 11. Clean column names
df_copy.columns = df_copy.columns.str.strip()
df_copy = df_copy.rename(columns={
    "order_id": "Order_ID",
    "order_date": "Order_Date",
    "customer_id": "Customer_ID",
    "product_category": "Product_Category",
    "region": "Region",
    "quantity": "Quantity",
    "unit_price": "Unit_Price",
    "discount": "Discount",
    "payment_method": "Payment_Method",
    "delivery_days": "Delivery_Days",
    "customer_rating": "Customer_Rating",
    "revenue": "Revenue"
})
print(df_copy.columns)

# 12. Remove extra spaces from Product Category
df_copy["Product_Category"] = df_copy["Product_Category"].str.strip()

# 13. Remove extra spaces from Region
df_copy["Region"] = df_copy["Region"].str.strip()

# 14. Remove extra spaces from Payment Method
df_copy["Payment_Method"] = df_copy["Payment_Method"].str.strip()

# 15. Standardize Product Category
df_copy["Product_Category"] = df_copy["Product_Category"].str.title()

# 16. Standardize Region
df_copy["Region"] = df_copy["Region"].str.title()

# 17. Standardize Payment Method
df_copy["Payment_Method"] = df_copy["Payment_Method"].str.title()
df_copy["Payment_Method"] = df_copy["Payment_Method"].replace({"Cod": "COD"})

# 18. Convert numeric columns
numeric_cols = [
    "Quantity",
    "Unit_Price",
    "Discount",
    "Delivery_Days",
    "Customer_Rating",
    "Revenue"
]

for col in numeric_cols:
    df_copy[col] = pd.to_numeric(
        df_copy[col], errors="coerce"
    )

# 19. Convert Order Date into datetime (month/day/year)
df_copy["Order_Date"] = pd.to_datetime(
    df_copy["Order_Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 20. Check missing values after conversion
print(df_copy.isnull().sum())

# 21. Fill missing numeric values with median
for col in numeric_cols:
    df_copy[col] = df_copy[col].fillna(
        df_copy[col].median()
    )

# 22. Fill missing Product Category, Region and Payment Method
for col in ["Product_Category", "Region", "Payment_Method"]:
    df_copy[col] = df_copy[col].fillna("Unknown")

# 23. Remove rows with missing ID or Order Date
df_copy = df_copy.dropna(
    subset=["Order_ID", "Customer_ID", "Order_Date"]
)

# 24. Check duplicate Order ID (every order should be unique)
print("Duplicate Order_ID:", df_copy["Order_ID"].duplicated().sum())

# 25. Find invalid Quantity
print(df_copy[df_copy["Quantity"] <= 0])

# 26. Find invalid Unit Price
print(df_copy[df_copy["Unit_Price"] <= 0])

# 27. Find invalid Discount (should be between 0 and 1)
print(df_copy[
    (df_copy["Discount"] < 0) |
    (df_copy["Discount"] > 1)
])

# 28. Find invalid Customer Rating (should be between 1 and 5)
print(df_copy[
    (df_copy["Customer_Rating"] < 1) |
    (df_copy["Customer_Rating"] > 5)
])

# 29. Find negative Delivery Days
print(df_copy[df_copy["Delivery_Days"] < 0])

# 30. Find negative Revenue
print(df_copy[df_copy["Revenue"] < 0])

# 31. Check future Order Dates (reported only, rows are kept)
future_dates = df_copy[df_copy["Order_Date"] > pd.Timestamp.today()]
print("Future Order_Date rows:", len(future_dates))
print("Date range:", df_copy["Order_Date"].min(), "to", df_copy["Order_Date"].max())

# 32. Calculate Revenue
df_copy["Calculated_Revenue"] = (
    df_copy["Quantity"] *
    df_copy["Unit_Price"] *
    (1 - df_copy["Discount"])
).round(2)

# 33. Calculate Revenue Difference
df_copy["Revenue_Difference"] = (
    df_copy["Revenue"] -
    df_copy["Calculated_Revenue"]
).round(2)

# 34. Count rows where Revenue is wrong
print(
    "Wrong Revenue rows:",
    (df_copy["Revenue_Difference"].abs() > 0.02).sum()
)

# 35. Find outliers using IQR
for col in numeric_cols:

    Q1 = df_copy[col].quantile(0.25)
    Q3 = df_copy[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df_copy[
        (df_copy[col] < lower_limit) |
        (df_copy[col] > upper_limit)
    ]

    print(col, "Outliers:", len(outliers))

# 36. Convert data types
df_copy["Order_ID"] = df_copy["Order_ID"].astype(int)
df_copy["Customer_ID"] = df_copy["Customer_ID"].astype(int)
df_copy["Quantity"] = df_copy["Quantity"].astype(int)
df_copy["Delivery_Days"] = df_copy["Delivery_Days"].astype(int)

for col in ["Product_Category", "Region", "Payment_Method"]:
    df_copy[col] = df_copy[col].astype("category")

# 37. Create Year, Quarter, Month and Weekday columns
df_copy["Year"] = df_copy["Order_Date"].dt.year
df_copy["Quarter"] = df_copy["Order_Date"].dt.quarter
df_copy["Month"] = df_copy["Order_Date"].dt.month
df_copy["Month_Name"] = df_copy["Order_Date"].dt.month_name()
df_copy["Weekday"] = df_copy["Order_Date"].dt.day_name()

# 38. Create Discount Percentage
df_copy["Discount_Pct"] = (df_copy["Discount"] * 100).round(0).astype(int)

# 39. Create Discount Amount
df_copy["Discount_Amount"] = (
    df_copy["Quantity"] * df_copy["Unit_Price"] -
    df_copy["Revenue"]
).round(2)

# 40. Sort values
df_copy = df_copy.sort_values(
    by=["Order_Date", "Order_ID"]
)

# 41. Reset index
df_copy = df_copy.reset_index(drop=True)

# 42. Final shape
print(df_copy.shape)

# 43. Final missing values
print(df_copy.isnull().sum())

# 44. Final duplicate rows
print(df_copy.duplicated().sum())

# 45. Display cleaned dataset
print(df_copy.head())

# 46. Save cleaned dataset
df_copy.to_csv(
    "ecommerce_sales_cleaned.csv",
    index=False
)

print("Data Cleaning Completed Successfully!")


   order_id order_date  customer_id product_category region  quantity  \
0     10001   1/1/2022         1102           Beauty  South         7   
1     10002   1/2/2022         1435         Clothing  South         7   
2     10003   1/3/2022         1860           Beauty   East         3   
3     10004   1/4/2022         1270      Electronics   West         5   
4     10005   1/5/2022         1106         Clothing   West         5   

   unit_price  discount payment_method  delivery_days  customer_rating  \
0      373.65      0.28         Wallet             10              4.7   
1       47.74      0.09           Card              6              3.9   
2      311.28      0.31            COD              6              2.5   
3      524.47      0.02         Wallet              6              1.6   
4      139.87      0.33         Wallet              4              4.9   

   revenue  
0  1883.20  
1   304.10  
2   644.35  
3  2569.90  
4   468.56  
      order_id order_date  customer_id

In [4]:


# 1. Find total revenue for each product_category.
print(df.groupby('product_category')['revenue'].sum())

# 2. Find average revenue for each product_category.
print(df.groupby('product_category')['revenue'].mean())

# 3. Find maximum revenue for each product_category.
print(df.groupby('product_category')['revenue'].max())

# 4. Find minimum revenue for each product_category.
print(df.groupby('product_category')['revenue'].min())

# 5. Count total orders for each product_category.
print(df.groupby('product_category')['order_id'].count())

# 6. Find total quantity sold for each product_category.
print(df.groupby('product_category')['quantity'].sum())

# 7. Find average quantity sold for each product_category.
print(df.groupby('product_category')['quantity'].mean())

# 8. Find maximum quantity sold for each product_category.
print(df.groupby('product_category')['quantity'].max())

# 9. Find minimum quantity sold for each product_category.
print(df.groupby('product_category')['quantity'].min())

# 10. Find average unit price for each product_category.
print(df.groupby('product_category')['unit_price'].mean())

# 11. Find total revenue for each region.
print(df.groupby('region')['revenue'].sum())

# 12. Find average revenue for each region.
print(df.groupby('region')['revenue'].mean())

# 13. Find maximum revenue for each region.
print(df.groupby('region')['revenue'].max())

# 14. Find minimum revenue for each region.
print(df.groupby('region')['revenue'].min())

# 15. Count total orders for each region.
print(df.groupby('region')['order_id'].count())

# 16. Find total quantity sold in each region.
print(df.groupby('region')['quantity'].sum())

# 17. Find average delivery days for each region.
print(df.groupby('region')['delivery_days'].mean())

# 18. Find maximum delivery days for each region.
print(df.groupby('region')['delivery_days'].max())

# 19. Find average customer rating for each region.
print(df.groupby('region')['customer_rating'].mean())

# 20. Find average discount for each region.
print(df.groupby('region')['discount'].mean())




# 21. Find total revenue for each payment_method.
print(df.groupby('payment_method')['revenue'].sum())

# 22. Find average revenue for each payment_method.
print(df.groupby('payment_method')['revenue'].mean())

# 23. Count orders for each payment_method.
print(df.groupby('payment_method')['order_id'].count())

# 24. Find total quantity sold for each payment_method.
print(df.groupby('payment_method')['quantity'].sum())

# 25. Find average quantity for each payment_method.
print(df.groupby('payment_method')['quantity'].mean())

# 26. Find maximum revenue for each payment_method.
print(df.groupby('payment_method')['revenue'].max())

# 27. Find minimum revenue for each payment_method.
print(df.groupby('payment_method')['revenue'].min())

# 28. Find average discount for each payment_method.
print(df.groupby('payment_method')['discount'].mean())

# 29. Find average customer rating for each payment_method.
print(df.groupby('payment_method')['customer_rating'].mean())

# 30. Find average delivery days for each payment_method.
print(df.groupby('payment_method')['delivery_days'].mean())




# 31. Find total revenue generated by each customer.
print(df.groupby('customer_id')['revenue'].sum())

# 32. Find average revenue for each customer.
print(df.groupby('customer_id')['revenue'].mean())

# 33. Count total orders placed by each customer.
print(df.groupby('customer_id')['order_id'].count())

# 34. Find total quantity purchased by each customer.
print(df.groupby('customer_id')['quantity'].sum())

# 35. Find maximum purchase revenue for each customer.
print(df.groupby('customer_id')['revenue'].max())

# 36. Find minimum purchase revenue for each customer.
print(df.groupby('customer_id')['revenue'].min())

# 37. Find average customer rating for each customer.
print(df.groupby('customer_id')['customer_rating'].mean())

# 38. Find average discount received by each customer.
print(df.groupby('customer_id')['discount'].mean())

# 39. Find average delivery days for each customer.
print(df.groupby('customer_id')['delivery_days'].mean())

# 40. Find total spending and order count for each customer.
print(df.groupby('customer_id').agg(
    Total_Revenue=('revenue', 'sum'),
    Total_Orders=('order_id', 'count')
))



# 41. Find total, average, maximum and minimum revenue
# for each product_category.
print(df.groupby('product_category')['revenue'].agg(
    ['sum', 'mean', 'max', 'min']
))

# 42. Find total revenue, average revenue and total quantity
# for each region.
print(df.groupby('region').agg(
    Total_Revenue=('revenue', 'sum'),
    Average_Revenue=('revenue', 'mean'),
    Total_Quantity=('quantity', 'sum')
))

# 43. Find total orders, average discount and average rating
# for each product_category.
print(df.groupby('product_category').agg(
    Total_Orders=('order_id', 'count'),
    Average_Discount=('discount', 'mean'),
    Average_Rating=('customer_rating', 'mean')
))

# 44. Find total quantity, maximum quantity and minimum quantity
# for each region.
print(df.groupby('region')['quantity'].agg(
    ['sum', 'max', 'min']
))

# 45. Find average unit price, maximum price and minimum price
# for each product_category.
print(df.groupby('product_category')['unit_price'].agg(
    ['mean', 'max', 'min']
))

# 46. Find total revenue, total orders and average delivery days
# for each payment_method.
print(df.groupby('payment_method').agg(
    Total_Revenue=('revenue', 'sum'),
    Total_Orders=('order_id', 'count'),
    Average_Delivery=('delivery_days', 'mean')
))

# 47. Find average rating, maximum rating and minimum rating
# for each product_category.
print(df.groupby('product_category')['customer_rating'].agg(
    ['mean', 'max', 'min']
))

# 48. Find total revenue, average quantity and average discount
# for each region.
print(df.groupby('region').agg(
    Total_Revenue=('revenue', 'sum'),
    Average_Quantity=('quantity', 'mean'),
    Average_Discount=('discount', 'mean')
))

# 49. Find total orders, total quantity and total revenue
# for each customer.
print(df.groupby('customer_id').agg(
    Total_Orders=('order_id', 'count'),
    Total_Quantity=('quantity', 'sum'),
    Total_Revenue=('revenue', 'sum')
))

# 50. Find average revenue, average rating and average delivery
# for each payment_method.
print(df.groupby('payment_method').agg(
    Average_Revenue=('revenue', 'mean'),
    Average_Rating=('customer_rating', 'mean'),
    Average_Delivery=('delivery_days', 'mean')))


# 51. Find total revenue for each region and product_category.
print(df.groupby(
    ['region', 'product_category']
)['revenue'].sum())

# 52. Find average revenue for each region and product_category.
print(df.groupby(
    ['region', 'product_category']
)['revenue'].mean())

# 53. Count orders for each region and payment_method.
print(df.groupby(
    ['region', 'payment_method']
)['order_id'].count())

# 54. Find total quantity for each region and product_category.
print(df.groupby(
    ['region', 'product_category']
)['quantity'].sum())

# 55. Find average discount for each product_category
# and payment_method.
print(df.groupby(
    ['product_category', 'payment_method']
)['discount'].mean())

# 56. Find average customer rating for each region
# and product_category.
print(df.groupby(
    ['region', 'product_category']
)['customer_rating'].mean())

# 57. Find total revenue for each product_category
# and payment_method.
print(df.groupby(
    ['product_category', 'payment_method']
)['revenue'].sum())

# 58. Find average delivery days for each region
# and payment_method.
print(df.groupby(
    ['region', 'payment_method']
)['delivery_days'].mean())

# 59. Find total revenue and total quantity for each
# region and payment_method.
print(df.groupby(
    ['region', 'payment_method']
).agg(
    Total_Revenue=('revenue', 'sum'),
    Total_Quantity=('quantity', 'sum')
))

# 60. Find total orders, revenue and average rating for
# each region and product_category.
print(df.groupby(
    ['region', 'product_category']
).agg(
    Total_Orders=('order_id', 'count'),
    Total_Revenue=('revenue', 'sum'),
    Average_Rating=('customer_rating', 'mean')
))




# 61. Find the product_category with the highest total revenue.
result = df.groupby('product_category')['revenue'].sum()
print(result.sort_values(ascending=False))

# 62. Find the region with the highest total revenue.
result = df.groupby('region')['revenue'].sum()
print(result.sort_values(ascending=False))

# 63. Find the top 5 customers by total revenue.
result = df.groupby('customer_id')['revenue'].sum()
print(result.sort_values(ascending=False).head(5))

# 64. Find the top 5 customers by number of orders.
result = df.groupby('customer_id')['order_id'].count()
print(result.sort_values(ascending=False).head(5))

# 65. Find product categories with average revenue greater than 1000.
result = df.groupby('product_category')['revenue'].mean()
print(result[result > 1000])

# 66. Find regions where total revenue is greater than 500000.
result = df.groupby('region')['revenue'].sum()
print(result[result > 500000])

# 67. Find customers whose total spending is greater than 10000.
result = df.groupby('customer_id')['revenue'].sum()
print(result[result > 10000])

# 68. Find payment methods having more than 1000 orders.
result = df.groupby('payment_method')['order_id'].count()
print(result[result > 1000])

# 69. Find product categories having average customer rating
# greater than 3.5.
result = df.groupby('product_category')['customer_rating'].mean()
print(result[result > 3.5])

# 70. Find regions having average delivery days less than 7.
result = df.groupby('region')['delivery_days'].mean()
print(result[result < 7])



# Convert order_date into datetime format.
df['order_date'] = pd.to_datetime(df['order_date'])

# Create year and month columns.
df['Year'] = df['order_date'].dt.year
df['Month'] = df['order_date'].dt.month
df['Month_Name'] = df['order_date'].dt.month_name()

# 71. Find total revenue for each year.
print(df.groupby('Year')['revenue'].sum())

# 72. Find average revenue for each year.
print(df.groupby('Year')['revenue'].mean())

# 73. Count total orders for each year.
print(df.groupby('Year')['order_id'].count())

# 74. Find total revenue for each month.
print(df.groupby('Month')['revenue'].sum())

# 75. Find average revenue for each month.
print(df.groupby('Month')['revenue'].mean())

# 76. Find total quantity sold for each month.
print(df.groupby('Month')['quantity'].sum())

# 77. Find total revenue for each year and month.
print(df.groupby(
    ['Year', 'Month']
)['revenue'].sum())

# 78. Find total orders for each year and month.
print(df.groupby(
    ['Year', 'Month']
)['order_id'].count())

# 79. Find total revenue for each region and year.
print(df.groupby(
    ['region', 'Year']
)['revenue'].sum())

# 80. Find total revenue, total orders and average revenue
# for each month.
print(df.groupby('Month_Name').agg(
    Total_Revenue=('revenue', 'sum'),
    Total_Orders=('order_id', 'count'),
    Average_Revenue=('revenue', 'mean')
))

product_category
Beauty          765860.88
Clothing       1531931.72
Electronics    1829899.22
Home            982083.92
Name: revenue, dtype: float64
product_category
Beauty         1059.281992
Clothing       1000.608570
Electronics    1029.768835
Home           1013.502497
Name: revenue, dtype: float64
product_category
Beauty         4005.93
Clothing       3928.63
Electronics    4119.33
Home           3640.56
Name: revenue, dtype: float64
product_category
Beauty         20.94
Clothing       11.21
Electronics    13.26
Home           13.01
Name: revenue, dtype: float64
product_category
Beauty          723
Clothing       1531
Electronics    1777
Home            969
Name: order_id, dtype: int64
product_category
Beauty         2995
Clothing       6171
Electronics    7109
Home           3949
Name: quantity, dtype: int64
product_category
Beauty         4.142462
Clothing       4.030699
Electronics    4.000563
Home           4.075335
Name: quantity, dtype: float64
product_category
Beauty     